# 02. Kiểm tra hash đồ thị của src/build_graph.py (task 0.4, tiêu chí 1)

Dựng lại đồ thị NF-ToN-IoT-v2 (W = S = 1000 flow, chia `chronological` 70/15/15) bằng module và so với hash của notebook Tuần 1 (`cb8842d60f9a3a5a`, 16.941 đồ thị). Dữ liệu: `athena21/netflow-datasets-v2`, file `NF-ToN-IoT-v2.csv` 2.676.491.000 byte, trùng dung lượng file Phong dùng (2552,5 MB).

Phần dựng đồ thị chạy trong tiến trình con mới (`scripts/check_graph_hash.py`), vì `pip install` đè numpy trong kernel đang chạy làm hỏng import (lỗi ở bản 1 của notebook này).

In [ ]:
import glob, os, subprocess, sys

def run(cmd, **kw):
    r = subprocess.run(cmd, capture_output=True, text=True, **kw)
    print(r.stdout[-4000:], end="")
    print(r.stderr[-2000:], end="", file=sys.stderr)
    print(f"[mã thoát {r.returncode}]")
    if r.returncode != 0:
        raise RuntimeError(f"Lệnh lỗi: {' '.join(cmd)}")

REPO = "/kaggle/working/gnn-iot-ids"
run(["rm", "-rf", REPO])
run(["git", "clone", "--depth", "1", "https://github.com/minucedev/gnn-iot-ids.git", REPO])
run(["git", "-C", REPO, "log", "-1", "--format=%H %s"])
with open(f"{REPO}/requirements.txt") as f:
    reqs = [l.strip() for l in f if l.strip() and not l.startswith("#")]
keep = [r for r in reqs if r.split("==")[0].lower() not in {"torch", "jupyter", "ipykernel"}]
run([sys.executable, "-m", "pip", "install", "-q", *keep])

In [ ]:
CSV = next(f for f in glob.glob("/kaggle/input/**/NF-ToN-IoT-v2.csv", recursive=True))
print(CSV, os.path.getsize(CSV), "byte")
run([sys.executable, "-I", "scripts/check_graph_hash.py", "nf_ton_iot_v2", CSV, "1000", "1000", "cb8842d60f9a3a5a"], cwd=REPO)